# 03 - Fetch Games (Season Schedule)

**Database: MySQL** (`NHL_Analytics_Hub`). Run notebooks 01 and 02 first.

**Endpoint:** `GET https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/{season}`

Called once per team. Every game appears twice (home team's schedule and away team's schedule),
so rows are de-duplicated on `game_id` and loaded with an upsert. The upsert also refreshes
scores and game state for games that were loaded earlier, so no separate "refresh" step is needed.

In [1]:
%pip install requests pandas mysql-connector-python

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import re
import time
import requests
import pandas as pd
import mysql.connector
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from IPython.display import display

REQUEST_DELAY = 0.4

# Same settings as notebooks 01 and 02
DB_HOST     = os.getenv("DB_HOST", "127.0.0.1")
DB_PORT     = int(os.getenv("DB_PORT", "3306"))
DB_USER     = os.getenv("DB_USER", "root")
DB_PASSWORD = os.getenv("DB_PASSWORD", "root")   # <-- change to YOUR MySQL password
DB_NAME     = os.getenv("DB_NAME", "NHL_Analytics_Hub")

def get_connection():
    return mysql.connector.connect(
        host=DB_HOST, port=DB_PORT, user=DB_USER, password=DB_PASSWORD,
        database=DB_NAME, connection_timeout=15,
    )

session = requests.Session()
_retry = Retry(total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504],
               allowed_methods=["GET"], respect_retry_after_header=True)
session.mount("https://", HTTPAdapter(max_retries=_retry))

def get_json(url):
    resp = session.get(url, timeout=(10, 30))
    resp.raise_for_status()
    data = resp.json()
    if not isinstance(data, dict):
        raise ValueError("Unexpected API response")
    return data

In [3]:
def text_val(v, limit=None):
    if isinstance(v, dict):
        v = v.get("default")
    if v is None:
        return None
    v = str(v).strip()
    if not v:
        return None
    return v[:limit] if limit else v

def int_val(v):
    try:
        return int(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def float_val(v):
    try:
        return float(v) if v is not None else None
    except (TypeError, ValueError):
        return None

def date_val(v):
    return v if isinstance(v, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", v) else None

## Load the team list and the season from notebook 01

In [4]:
conn = get_connection()
cur = conn.cursor()

cur.execute("SELECT team_id, team_abbrev FROM teams ORDER BY team_abbrev")
teams_df = pd.DataFrame(cur.fetchall(), columns=["team_id", "team_abbrev"])
if teams_df.empty:
    raise RuntimeError("teams table is empty - run notebook 01 first.")
team_id_lookup = dict(zip(teams_df.team_abbrev, teams_df.team_id.astype(int)))

# Use the season notebook 01 loaded, instead of hardcoding it
cur.execute("SELECT MAX(season) FROM standings")
SEASON = cur.fetchone()[0] or "20252026"
SCHEDULE_URL = "https://api-web.nhle.com/v1/club-schedule-season/{team_abbrev}/" + SEASON

print(f"{len(teams_df)} teams loaded. Season: {SEASON}")
display(teams_df.head())

32 teams loaded. Season: 20262027


,team_id,team_abbrev
0,26,ANA
1,4,BOS
2,18,BUF
3,1,CAR
4,28,CBJ


## Create the `games` table

The original notebook never created it. `home_team_id` and `away_team_id` are both foreign keys to `teams`.

In [5]:
GAMES_SQL = """
CREATE TABLE IF NOT EXISTS games (
    game_id BIGINT PRIMARY KEY,
    season VARCHAR(20) NOT NULL,
    game_type INT,
    game_date DATE,
    home_team_id INT NOT NULL,
    away_team_id INT NOT NULL,
    home_score INT,
    away_score INT,
    game_state VARCHAR(20),
    venue_name VARCHAR(150),
    FOREIGN KEY (home_team_id) REFERENCES teams(team_id),
    FOREIGN KEY (away_team_id) REFERENCES teams(team_id),
    INDEX idx_games_date (game_date),
    INDEX idx_games_state (game_state)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
"""
cur.execute(GAMES_SQL)
conn.commit()
print("games table ready")

games table ready


## Inspect one schedule response before writing the parser

In [6]:
sample_abbrev = teams_df.team_abbrev.iloc[0]
sample_data = get_json(SCHEDULE_URL.format(team_abbrev=sample_abbrev))

print("Top-level keys:", list(sample_data.keys()))
print("Games in this team's season:", len(sample_data.get("games", [])))
display(sample_data["games"][0])

Top-level keys: ['previousSeason', 'currentSeason', 'howToWatch', 'clubTimezone', 'clubUTCOffset', 'games']
Games in this team's season: 88


{'id': 2026010009,
 'season': 20262027,
 'gameType': 1,
 'gameDate': '2026-09-20',
 'venue': {'default': 'Honda Center'},
 'neutralSite': False,
 'startTimeUTC': '2026-09-20T20:00:00Z',
 'easternUTCOffset': '-04:00',
 'venueUTCOffset': '-07:00',
 'venueTimezone': 'America/Los_Angeles',
 'gameState': 'FINAL',
 'gameScheduleState': 'OK',
 'tvBroadcasts': [],
 'awayTeam': {'id': 28,
  'commonName': {'default': 'Sharks'},
  'placeName': {'default': 'San Jose'},
  'placeNameWithPreposition': {'default': 'San Jose', 'fr': 'de San Jose'},
  'abbrev': 'SJS',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/SJS_light.svg',
  'darkLogo': 'https://assets.nhle.com/logos/nhl/svg/SJS_dark.svg',
  'awaySplitSquad': False,
  'score': 2},
 'homeTeam': {'id': 24,
  'commonName': {'default': 'Ducks'},
  'placeName': {'default': 'Anaheim'},
  'placeNameWithPreposition': {'default': 'Anaheim', 'fr': "d'Anaheim"},
  'abbrev': 'ANA',
  'logo': 'https://assets.nhle.com/logos/nhl/svg/ANA_light.svg',
  'darkLog

## Parse one team's schedule

Scores are `None` until a game is played. They are kept as `None` (not pandas `NaN`), which MySQL stores as `NULL`.

In [7]:
def parse_schedule(schedule_json, team_id_lookup):
    games = []
    for g in schedule_json.get("games") or []:
        home = g.get("homeTeam") or {}
        away = g.get("awayTeam") or {}
        home_abbrev = (home.get("abbrev") or "").upper()
        away_abbrev = (away.get("abbrev") or "").upper()

        # Skip games vs non-NHL opponents (e.g. some preseason games)
        if g.get("id") is None or home_abbrev not in team_id_lookup or away_abbrev not in team_id_lookup:
            continue

        games.append({
            "game_id": int(g["id"]),
            "season": text_val(g.get("season"), 20) or SEASON,
            "game_type": int_val(g.get("gameType")),
            "game_date": date_val(g.get("gameDate")),
            "home_team_id": team_id_lookup[home_abbrev],
            "away_team_id": team_id_lookup[away_abbrev],
            "home_score": int_val(home.get("score")),
            "away_score": int_val(away.get("score")),
            "game_state": text_val(g.get("gameState"), 20),
            "venue_name": text_val((g.get("venue") or {}).get("default"), 150),
        })
    return games

test_games = parse_schedule(sample_data, team_id_lookup)
print(f"Parsed {len(test_games)} games for {sample_abbrev}")
display(pd.DataFrame(test_games).head())

Parsed 88 games for ANA


,game_id,season,game_type,game_date,home_team_id,away_team_id,home_score,away_score,game_state,venue_name
0,2026010009,20262027,1,2026-09-20,26,21,6.0,2.0,FINAL,Honda Center
1,2026010033,20262027,1,2026-09-23,26,29,1.0,2.0,FINAL,Honda Center
2,2026010039,20262027,1,2026-09-24,21,26,10.0,0.0,FINAL,SAP Center at San Jose
3,2026010052,20262027,1,2026-09-26,29,26,6.0,2.0,FINAL,Crypto.com Arena
4,2026020021,20262027,2,2026-10-02,5,26,3.0,4.0,OFF,T-Mobile Arena


## Loop over all teams (with one retry pass)

In [8]:
all_games = []

def collect(abbrev):
    parsed = parse_schedule(get_json(SCHEDULE_URL.format(team_abbrev=abbrev)), team_id_lookup)
    all_games.extend(parsed)
    return len(parsed)

failed_teams = []
for abbrev in teams_df.team_abbrev:
    try:
        print(f"{abbrev}: OK ({collect(abbrev)} games)")
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"{abbrev}: FAILED - {e}")
        failed_teams.append(abbrev)
    time.sleep(REQUEST_DELAY)

still_failed = []
for abbrev in failed_teams:
    time.sleep(2)
    try:
        print(f"{abbrev}: retry OK ({collect(abbrev)} games)")
    except (requests.exceptions.RequestException, ValueError) as e:
        print(f"{abbrev}: retry FAILED - {e}")
        still_failed.append(abbrev)

print(f"\nGame rows collected (with duplicates across both teams): {len(all_games)}")
if still_failed:
    raise RuntimeError(f"Teams still failing: {still_failed}. Nothing was saved; re-run this cell.")

ANA: OK (88 games)
BOS: OK (88 games)
BUF: OK (88 games)
CAR: OK (88 games)
CBJ: OK (88 games)
CGY: OK (88 games)
CHI: OK (88 games)
COL: OK (88 games)
DAL: OK (88 games)
DET: OK (88 games)
EDM: OK (88 games)
FLA: OK (88 games)
LAK: OK (88 games)
MIN: OK (88 games)
MTL: OK (89 games)
NJD: OK (88 games)
NSH: OK (88 games)
NYI: OK (88 games)
NYR: OK (88 games)
OTT: OK (89 games)
PHI: OK (88 games)
PIT: OK (88 games)
SEA: OK (88 games)
SJS: OK (88 games)
STL: OK (88 games)
TBL: OK (88 games)
TOR: OK (88 games)
UTA: OK (88 games)
VAN: OK (88 games)
VGK: OK (88 games)
WPG: OK (88 games)
WSH: OK (88 games)

Game rows collected (with duplicates across both teams): 2818


## De-duplicate on `game_id`

In [9]:
unique = {}
for g in all_games:
    unique[g["game_id"]] = g
games = list(unique.values())
print(f"{len(all_games)} total rows -> {len(games)} unique games after dedup")
display(pd.DataFrame(games).head())

2818 total rows -> 1409 unique games after dedup


,game_id,season,game_type,game_date,home_team_id,away_team_id,home_score,away_score,game_state,venue_name
0,2026010009,20262027,1,2026-09-20,26,21,6.0,2.0,FINAL,Honda Center
1,2026010033,20262027,1,2026-09-23,26,29,1.0,2.0,FINAL,Honda Center
2,2026010039,20262027,1,2026-09-24,21,26,10.0,0.0,FINAL,SAP Center at San Jose
3,2026010052,20262027,1,2026-09-26,29,26,6.0,2.0,FINAL,Crypto.com Arena
4,2026020021,20262027,2,2026-10-02,5,26,3.0,4.0,OFF,T-Mobile Arena


## Load into the `games` table (upsert, one transaction)

`ON DUPLICATE KEY UPDATE` inserts new games and refreshes scores/state of existing ones.

In [10]:
COLUMNS = ["game_id", "season", "game_type", "game_date", "home_team_id", "away_team_id",
           "home_score", "away_score", "game_state", "venue_name"]

UPSERT_SQL = f"""
INSERT INTO games ({", ".join(COLUMNS)})
VALUES ({", ".join(["%s"] * len(COLUMNS))})
ON DUPLICATE KEY UPDATE
    {", ".join(f"{c}=VALUES({c})" for c in COLUMNS[1:])}
"""

rows = [tuple(g[c] for c in COLUMNS) for g in games]

try:
    conn.start_transaction()
    cur.executemany(UPSERT_SQL, rows)
    conn.commit()
    cur.execute("SELECT COUNT(*) FROM games")
    print("Rows in games:", cur.fetchone()[0])
except Exception:
    conn.rollback()
    raise

Rows in games: 1409


## Sanity checks

In [11]:
cur.execute("SELECT game_state, COUNT(*) AS game_count FROM games GROUP BY game_state ORDER BY game_count DESC")
display(pd.DataFrame(cur.fetchall(), columns=["game_state", "game_count"]))

,game_state,game_count
0,FUT,1128
1,OFF,216
2,FINAL,65


In [12]:
cur.execute("""
    SELECT t.team_abbrev, COUNT(*) AS games_scheduled
    FROM teams t
    JOIN games g ON t.team_id = g.home_team_id OR t.team_id = g.away_team_id
    GROUP BY t.team_abbrev
    ORDER BY games_scheduled ASC
""")
display(pd.DataFrame(cur.fetchall(), columns=["team_abbrev", "games_scheduled"]).head(10))

,team_abbrev,games_scheduled
0,ANA,88
1,BOS,88
2,BUF,88
3,CAR,88
4,CBJ,88
5,CGY,88
6,CHI,88
7,COL,88
8,DAL,88
9,DET,88


In [13]:
# Finished games are 'OFF' or 'FINAL' (FINAL is used shortly after the game ends)
cur.execute("""
    SELECT g.game_date, ht.team_abbrev AS home, g.home_score,
           aw.team_abbrev AS away, g.away_score, g.game_state
    FROM games g
    JOIN teams ht ON g.home_team_id = ht.team_id
    JOIN teams aw ON g.away_team_id = aw.team_id
    WHERE g.game_state IN ('OFF', 'FINAL')
    ORDER BY g.game_date DESC
    LIMIT 10
""")
cols = ["game_date", "home", "home_score", "away", "away_score", "game_state"]
display(pd.DataFrame(cur.fetchall(), columns=cols))

,game_date,home,home_score,away,away_score,game_state
0,2026-10-30,CGY,6,SEA,2,OFF
1,2026-10-30,FLA,5,WPG,2,OFF
2,2026-10-30,VGK,2,NJD,5,OFF
3,2026-10-29,SJS,3,VAN,2,OFF
4,2026-10-29,LAK,6,OTT,5,OFF
5,2026-10-29,ANA,6,BUF,7,OFF
6,2026-10-29,MIN,3,NYI,4,OFF
7,2026-10-29,DAL,2,MTL,5,OFF
8,2026-10-29,NSH,3,WSH,4,OFF
9,2026-10-29,STL,4,BOS,1,OFF


In [14]:
cur.close()
conn.close()
session.close()